# 01 · How long until the next sale? (Survival analysis)

**Question:** after a Steam game's sale ends, how long until it goes on sale again?

This is the same structure as the customer-churn example in **CPSC 330 Lecture 20 (Survival analysis)**:

| Churn lecture | This project |
|---|---|
| a customer | a gap between two sales of one game |
| `tenure` (months stayed) | `duration_days` (days until the next sale) |
| `Churn = Yes` | `event_observed = True` (the next sale happened) |
| customer still subscribed (right-censored) | game hasn't gone on sale again *yet* |

Plan, following the course:
1. Look at the data and the amount of censoring.
2. Show why plain regression on `duration_days` is biased (the lecture's "naive fixes that fail").
3. **Kaplan-Meier** curves: overall and by group.
4. **Cox proportional hazards** with features, evaluated with the **concordance index**.
5. Compare against baselines (random, and "just wait for the next seasonal sale").
6. Forward-looking predictions with `conditional_after`: *"it's been 40 days, what's the chance of a sale in the next 14?"*

Data: `fct_sale_gaps` and `dim_games` from the dbt warehouse (run `python scripts/run_dbt.py build` first).

In [1]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.utils import concordance_index
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (9, 5)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
con = duckdb.connect(str(ROOT / "data" / "warehouse.duckdb"), read_only=True)

ImportError: DLL load failed while importing _odepack: An Application Control policy has blocked this file.

## 1. Load the data

One row per gap. We keep only **modelable** games (released, for sale, not free, not adult) and add the game's age at the start of each gap. Every feature is something we'd know **on the day the gap starts**, so nothing leaks from the future.

In [ ]:
gaps = con.sql("""
    select
        g.*,
        d.name,
        d.main_publisher,
        d.review_count_proxy,
        date_diff('day', d.release_date, g.gap_start) / 365.25 as game_age_years
    from fct_sale_gaps g
    join dim_games d using (appid)
    where d.is_modelable
""").df()

gaps["gap_start"] = pd.to_datetime(gaps["gap_start"])
gaps["regular_dollars"] = gaps["regular_cents"] / 100
print(f"{len(gaps):,} gaps across {gaps['appid'].nunique():,} games")
gaps.head()

In [ ]:
print(f"Next sale observed: {gaps['event_observed'].mean():.1%}")
print(f"Right-censored (no next sale yet): {(~gaps['event_observed']).mean():.1%}")
gaps["duration_days"].describe()

In [ ]:
ax = gaps.loc[gaps["event_observed"], "duration_days"].clip(upper=730).hist(bins=60)
ax.set(title="Days between sales (observed gaps, capped at 2 years)", xlabel="days", ylabel="gaps");

## 2. Train/test split: by time, never shuffled

From **Lecture 19**: *"If we want to forecast, we aren't allowed to know what happened in the future."*

- **Train:** gaps that start before `CUTOFF`.
- **Test:** gaps that start on or after `CUTOFF`.

There's one extra survival-specific step. A training gap that starts *before* the cutoff but ends *after* it would leak future information through its outcome. So training gaps are **censored at the cutoff**: if the next sale came after `CUTOFF`, all we "knew" at the time is that the gap lasted at least until `CUTOFF`. This is called administrative censoring.

In [ ]:
CUTOFF = pd.Timestamp("2025-01-01")

train = gaps[gaps["gap_start"] < CUTOFF].copy()
test = gaps[gaps["gap_start"] >= CUTOFF].copy()

# administrative censoring at the cutoff for the training set
days_to_cutoff = (CUTOFF - train["gap_start"]).dt.days
ends_after_cutoff = train["duration_days"] > days_to_cutoff
train.loc[ends_after_cutoff, "event_observed"] = False
train.loc[ends_after_cutoff, "duration_days"] = days_to_cutoff[ends_after_cutoff]

print(f"train: {len(train):,} gaps ({train['event_observed'].mean():.1%} observed)")
print(f"test:  {len(test):,} gaps ({test['event_observed'].mean():.1%} observed)")

## 3. Why not just average the gap lengths?

The lecture's "naive fixes that fail", applied to our data:

- **Drop the censored gaps** and average the rest. This is biased **low**: games that go a long time without a sale are exactly the ones still waiting, so they get dropped.
- **Treat censored gaps as if the sale happened today.** Also biased low, because it cuts every waiting gap short.

Kaplan-Meier handles censoring properly.

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(train["duration_days"], event_observed=train["event_observed"], label="all games")

naive_drop = train.loc[train["event_observed"], "duration_days"].median()
naive_now = train["duration_days"].median()

pd.DataFrame({
    "median days until next sale": [naive_drop, naive_now, kmf.median_survival_time_],
}, index=["naive: drop censored", "naive: censored = sale today", "Kaplan-Meier"])

## 4. Kaplan-Meier curves

The y-axis is the probability that the game is **still not on sale** after *x* days. Steep drops mean sales tend to come at that point.

In [ ]:
ax = kmf.plot_survival_function()
ax.set(title="Time until next sale: all games (train)",
       xlabel="days since last sale ended", ylabel="P(still no sale)", xlim=(0, 730))
for d in (30, 90, 180, 365):
    print(f"P(sale within {d:>3} days) = {1 - kmf.predict(d):.1%}")

### By price tier

Do cheap games go on sale again sooner than expensive ones? Like the lecture's `SeniorCitizen` comparison, we fit one curve per group.

In [ ]:
train["price_tier"] = pd.cut(train["regular_dollars"], bins=[0, 10, 30, 1000],
                             labels=["under $10", "$10-30", "over $30"])

ax = plt.subplot()
for tier, grp in train.groupby("price_tier", observed=True):
    KaplanMeierFitter().fit(grp["duration_days"], grp["event_observed"], label=str(tier)) \
        .plot_survival_function(ax=ax)
ax.set(title="Time until next sale, by regular price", xlabel="days since last sale ended",
       ylabel="P(still no sale)", xlim=(0, 730));

### By how deep the last sale was

In [ ]:
train["prev_cut_tier"] = pd.cut(train["prev_sale_cut"], bins=[0, 30, 60, 100],
                                labels=["shallow (<=30%)", "medium (31-60%)", "deep (>60%)"])

ax = plt.subplot()
for tier, grp in train.groupby("prev_cut_tier", observed=True):
    KaplanMeierFitter().fit(grp["duration_days"], grp["event_observed"], label=str(tier)) \
        .plot_survival_function(ax=ax)
ax.set(title="Time until next sale, by depth of the previous sale", xlabel="days since last sale ended",
       ylabel="P(still no sale)", xlim=(0, 730));

## 5. Cox proportional hazards

Kaplan-Meier compares a handful of groups. **Cox** uses all the features at once, a bit like linear regression for survival.

- **Positive coefficient** → higher "hazard" → the next sale comes **sooner**
- **Negative coefficient** → the next sale comes **later**

(In the churn lecture, churn was the bad event. Here the event is a sale, so a higher hazard is good news for the buyer.)

Preprocessing follows **Lectures 5–6**: a `ColumnTransformer` that imputes and scales numeric features. We fit it on **train only**, then apply it to test.

In [ ]:
numeric_feats = [
    "regular_dollars",           # price tier
    "prev_sale_cut",             # how deep the last sale went
    "prev_sale_length_days",     # how long the last sale ran
    "prev_gap_days",             # the gap before the last sale (missing for a game's first sale)
    "days_to_next_major_event",  # Steam announces its calendar ahead, so this is known
    "game_age_years",
    "after_sale_number",         # how many sales the game has had so far
]

preprocessor = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), numeric_feats),
)

def to_lifelines_frame(df, fit=False):
    X = preprocessor.fit_transform(df) if fit else preprocessor.transform(df)
    cols = [c.split("__")[-1] for c in preprocessor.get_feature_names_out()]
    out = pd.DataFrame(X, columns=cols, index=df.index)
    out["duration_days"] = df["duration_days"].clip(lower=0.5)   # lifelines needs durations > 0
    out["event_observed"] = df["event_observed"].astype(int)
    return out

train_cox = to_lifelines_frame(train, fit=True)
test_cox = to_lifelines_frame(test)
train_cox.head()

In [ ]:
cph = CoxPHFitter(penalizer=0.1)   # same penalizer as the lecture
cph.fit(train_cox, duration_col="duration_days", event_col="event_observed")
cph.summary[["coef", "exp(coef)", "p"]].sort_values("coef")

In [ ]:
ax = cph.plot()
ax.set_title("Cox coefficients (right = next sale comes sooner)");

## 6. How good is it? Concordance index vs baselines

The **concordance index** (c-index) asks: for two gaps, does the model correctly say which one ends sooner? **0.5 = random, 1.0 = perfect.** It's the survival version of accuracy from the lecture.

Following **Lectures 2 and 11**, we compare against baselines in a `results` table:
- **Dummy:** random ordering (c-index 0.5)
- **Seasonal baseline:** *"the next sale comes with the next big Steam sale"*, i.e. rank gaps purely by `days_to_next_major_event`

In [ ]:
def cindex(df, predicted_days):
    # higher predicted duration should go with higher actual duration
    return concordance_index(df["duration_days"], predicted_days, df["event_observed"])

results = {}
results["Dummy (random)"] = {"train": 0.5, "test": 0.5}
results["Seasonal baseline"] = {
    "train": cindex(train, train["days_to_next_major_event"].fillna(999)),
    "test": cindex(test, test["days_to_next_major_event"].fillna(999)),
}
results["Cox PH"] = {
    "train": cph.score(train_cox, scoring_method="concordance_index"),
    "test": cph.score(test_cox, scoring_method="concordance_index"),
}
pd.DataFrame(results).T.round(3)

## 7. Forward-looking predictions: "should I wait?"

`predict_survival_function(..., conditional_after=...)` predicts from **today**, given how long the game has *already* gone without a sale. That's exactly the buyer's question.

For a few games from the test set, *"it's been N days since the last sale. What's the chance of a sale in the next 14 / 30 days?"*

In [ ]:
examples = test.sample(5, random_state=330)
already_waited = np.minimum(examples["duration_days"].to_numpy(), 40)

surv = cph.predict_survival_function(
    test_cox.loc[examples.index].drop(columns=["duration_days", "event_observed"]),
    times=[14, 30],
    conditional_after=already_waited,
)

pd.DataFrame({
    "game": examples["name"].to_numpy(),
    "regular $": examples["regular_dollars"].to_numpy(),
    "days since last sale": already_waited,
    "days to next big Steam sale": examples["days_to_next_major_event"].to_numpy(),
    "P(sale within 14 days)": (1 - surv.loc[14]).round(2).to_numpy(),
    "P(sale within 30 days)": (1 - surv.loc[30]).round(2).to_numpy(),
})

## 8. Takeaways (fill in after running on the real data)

- **Censoring matters:** the naive medians are clearly lower than the Kaplan-Meier median.
- **Price tier / previous depth:** _which groups go on sale again sooner?_
- **Cox coefficients:** _which features push the next sale sooner or later?_
- **C-index:** _did Cox beat the seasonal baseline on the test set, and by how much?_

**Limitations (worth saying in an interview):**
- Cox assumes **proportional hazards**. The lecture's `check_assumptions()` can test that, and seasonal effects probably break it, which is a reason to try tree models next.
- Features are fixed at the start of each gap. In reality, "days to the next big sale" changes as the gap goes on (time-varying covariates, mentioned at the end of the lecture).

**Next notebook:** a gradient-boosting classifier for *"sale within 14 days?"* (**Lecture 11**), explained with SHAP (**Lecture 12**), compared against this Cox model.